# 03 — Scattering, finite-time regions and absorption

A wave packet contains a distribution of momenta. Compare transmission with a weighted stationary prediction, retain still-interacting probability, and inspect the CAP budget independently.

The experiments below use **runner.run_simulation**, the same artifact-producing path used by the CLI and browser. A saved result is a measured configuration, not an automatic accuracy certificate.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "wavepacket").is_dir())
print("Python:", sys.executable)

OUT = ROOT / "results" / "notebook_runs" / "03_scattering_and_boundaries"
OUT.mkdir(parents=True, exist_ok=True)
import h5py
from wavepacket.config import SimulationConfig
from wavepacket.experiments import get_experiment
from wavepacket.model import make_grid, initial_state
from wavepacket.runner import run_simulation, read_observables
from wavepacket.validation import packet_transmission_reference

data = get_experiment("1d-barrier").model_dump()
data["sampling"].update(snapshots=5, observable_every=40)
barrier_config = SimulationConfig.model_validate(data)
barrier = run_simulation(barrier_config, OUT / "runs")
assert barrier["status"] == "completed"
folder = Path(barrier["path"])
grid = make_grid(barrier_config.grid)
reference = packet_transmission_reference(initial_state(barrier_config, grid), grid, [(2.5, 2.0)])
with h5py.File(folder / "state.h5", "r") as store:
    final = store["psi"][-1]
x = grid.axes[0]
probability = abs(final)**2 * grid.cell_volume
regions = {"reflected_region": float(probability[x < -3].sum()),
           "interaction_region": float(probability[abs(x) <= 3].sum()),
           "transmitted_region": float(probability[x > 3].sum())}
print({"stationary_packet_reference": reference, "finite_time_regions": regions,
       "remaining_norm": float(probability.sum())})
print("The reference/region discrepancy includes finite-grid, timestep and finite-time effects; inspect refinement evidence.")


In [ ]:
cap_config = SimulationConfig.model_validate({
    "name": "notebook-cap-ledger",
    "grid": {"bounds": [[-10, 10]], "points": [256]},
    "state": {"center": [5], "momentum": [3], "sigma": [1]},
    "solver": {"dt": .01, "duration": 2},
    "boundary": {"kind": "cap", "width": 3, "strength": 2, "power": 4},
    "sampling": {"snapshots": 4, "observable_every": 5}})
cap = run_simulation(cap_config, OUT / "runs")
assert cap["status"] == "completed"
rows = read_observables(Path(cap["path"]))
times = np.array([row["time"] for row in rows])
surviving = np.array([row["norm"] for row in rows])
absorbed = np.array([row["absorbed_probability"] for row in rows])
assert np.max(abs(surviving + absorbed - 1)) < 1e-10
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(x, abs(final)**2)
axes[0].axvspan(-3, 3, alpha=.15, color="grey", label="interaction partition")
axes[0].set(xlabel="x / L0", ylabel="density", title="Finite-time barrier result")
axes[0].legend()
axes[1].plot(times, surviving, label="surviving norm")
axes[1].plot(times, absorbed, label="CAP loss ledger")
axes[1].plot(times, surviving + absorbed, "--", label="sum")
axes[1].set(xlabel="t / t0", ylabel="probability", title="Explicit damping budget")
axes[1].legend()
fig.tight_layout()
fig.savefig(OUT / "scattering_and_cap.png", dpi=150)
plt.close(fig)
display(Image(filename=str(OUT / "scattering_and_cap.png")))
(OUT / "scattering_comparison.json").write_text(json.dumps({"reference": reference, "regions": regions,
    "barrier_run": barrier["id"], "cap_run": cap["id"]}, indent=2))


## 2D signed screen profiles and refinement

The release study varies spatial grid, timestep, domain size and CAP parameters separately for single and double slits. The plot below reads that saved empirical evidence rather than rerunning the full study. Profiles are signed time-integrated current, not detector counts, and are not independently normalised to look alike.

A missing release artifact is reported explicitly; generate it by running the full validation suite.

In [ ]:
study_path = ROOT / "results" / "validation" / "full" / "validation.json"
if not study_path.exists():
    print("Release screen study is absent. Run the full validation suite before claiming screen-profile convergence.")
else:
    import hashlib
    sys.path.insert(0, str(ROOT / "scripts"))
    from verify_release import SCIENCE_SOURCES
    full_report = json.loads(study_path.read_text(encoding="utf-8"))
    provenance = full_report["provenance"]
    hashes = provenance["source_sha256"]
    assert full_report["suite"] == "full" and full_report["status"] == "passed"
    assert not SCIENCE_SOURCES - hashes.keys()
    assert not provenance["sources_changed_during_run"]
    assert all(hashlib.sha256((ROOT / "src/wavepacket" / name).read_bytes()).hexdigest() == value
               for name, value in hashes.items()), "Saved screen evidence is stale; rerun full validation"
    screen_study = next(case for case in full_report["cases"]
                        if case["id"] == "published-slit-screen-profile-refinement")
    assert screen_study["status"] == "passed", screen_study
    studies = screen_study["metrics"]["studies"]
    fig, axes = plt.subplots(1, len(studies), figsize=(12, 4), squeeze=False)
    for ax, study in zip(axes[0], studies, strict=True):
        for name, profile in study["profiles"].items():
            ax.plot(profile["y"], profile["signed_integrated_current"], label=name, alpha=.85)
        ax.set(xlabel="y / L0", ylabel="signed integrated current [1/L0]", title=study["id"])
        ax.legend(fontsize=8)
        print(study["id"], "relative L1 changes:", study["relative_l1_differences"])
    fig.tight_layout()
    fig.savefig(OUT / "slit_screen_refinement.png", dpi=150)
    plt.close(fig)
    display(Image(filename=str(OUT / "slit_screen_refinement.png")))


## Review questions

1. Why is the reference integrated over positive momentum instead of evaluated at the packet's mean energy?
2. Why does a double-barrier packet need a residual interaction-region measurement?
3. Explain why this CAP budget can pass even if an absorber reflects too much. Which separate calibration tests address that?
4. Why is dividing the surviving state by its norm after every step physically misleading?
5. For 2D slits, explain the difference between density, signed current, and an actual detector model.

This notebook's CAP fixture validates its loss accounting. It does not claim universal reflection-free absorption.